# Lab 1 · Quantum coins & entangled pairs

**Qiskit Fall Fest 2026 @ UT San Antonio · Day 1 · Thursday, Nov 19, 2026**

You'll build three kinds of quantum "coins" and find out what makes them different from real ones:

| Part | You build | You find out |
|---|---|---|
| **A** | a quantum coin, $H\lvert0\rangle$ | why flipping it twice is *not* random |
| **B** | a biased coin, $R_y(\theta)\lvert0\rangle$ | how amplitudes set the odds, and what phase does |
| **C** | an entangled pair of coins | why entanglement is more than correlation |
| **D** | *bonus:* bigger and noisier circuits | a preview of Day 2 |

### How this notebook works

1. Write your code where it says `# YOUR CODE HERE`.
2. Run the cell, then run the **check** cell under it. It prints **correct! 🎉** or a hint.
3. Stuck? Open the **💡 hint** box under the task. Still stuck after 3 minutes? Skip ahead and come back.
4. At the end, run the **scoreboard** cell and post a screenshot in the event chat.
5. **Submit your completed lab by Sunday, Nov 22, 11:59 AM (Central Time)** with this form: [forms.gle/9h5PA4iqQg1wfGAcA](https://forms.gle/9h5PA4iqQg1wfGAcA)

This lab is running on simulator.

---
## 0 · Setup

Put your name below (it shows on your scoreboard), then run the two cells.

In [ ]:
YOUR_NAME = "Your Name"   # <- change this

In [ ]:
# %pip install --upgrade 'qiskit[visualization]' qiskit-aer qiskit-ibm-runtime -q   # only on your own laptop
from math import pi, sqrt, asin, sin
import numpy as np

import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.visualization import plot_histogram

sampler = StatevectorSampler()
estimator = StatevectorEstimator()
print("Qiskit", qiskit.__version__, "· ready!" if qiskit.__version__.startswith("2.") else "· please use Qiskit 2.x")

# Install and load the auto-checks for this lab
%pip install -q --no-deps --force-reinstall https://github.com/vantnprof/QFF2026_UTSA_grader/archive/refs/heads/main.zip
from qff_utsa_grader.lab1 import check, scoreboard

---
## Part A · The quantum coin

A real coin is heads *or* tails; we just don't know which. A **quantum coin** is a qubit in the superposition

$$
\lvert+\rangle = H\lvert 0\rangle = \tfrac{1}{\sqrt2}\lvert0\rangle + \tfrac{1}{\sqrt2}\lvert1\rangle ,
$$

and measuring it gives 0 or 1 with probability $\lvert\tfrac{1}{\sqrt2}\rvert^2 = \tfrac12$ each (the Born rule).

### A1 · Make a quantum coin

**Input:** a 1-qubit circuit `coin`. **Output:** its state should be $\lvert+\rangle$.

<details><summary><b>💡 Stuck? Click for a hint</b></summary>

One gate does it: the Hadamard, `coin.h(0)`.

</details>

In [ ]:
coin = QuantumCircuit(1)
# YOUR CODE HERE: put the qubit in superposition

Statevector(coin).draw("latex")

In [ ]:
check("A1")

### A2 · Flip it 1,000 times

**Input:** a copy of `coin` with measurements. **Output:** `coin_counts`, a dict like `{'0': 507, '1': 493}`.

Use the Sampler: `sampler.run([circuit], shots=1000).result()[0].data.meas.get_counts()`.

<details><summary><b>💡 Stuck? Click for a hint</b></summary>

`flips = coin.measure_all(inplace=False)` makes a measured copy.

Then `coin_counts = sampler.run([flips], shots=1000).result()[0].data.meas.get_counts()`.

</details>

In [ ]:
coin_counts = None
# YOUR CODE HERE: measure a copy of coin, run 1000 shots, store the counts

print(coin_counts)
if coin_counts:
    display(plot_histogram(coin_counts))

In [ ]:
check("A2")

### A3 · Flip it twice

Flip a real coin, then flip it again: still 50/50. Now "flip" the quantum coin twice by applying $H$ **twice**.

🤔 **Predict first:** what will the counts be?

**Input:** a 1-qubit circuit `double` with two Hadamards. **Output:** its state, and 1,000 measured flips.

<details><summary><b>💡 Click for a hint</b></summary>

`double.h(0)` two times.

Then compute $HH\lvert0\rangle$ by hand: $H$ is its own inverse!

</details>

In [ ]:
double = QuantumCircuit(1)
# YOUR CODE HERE: two Hadamards

print(Statevector(double))
print(sampler.run([double.measure_all(inplace=False)], shots=1000).result()[0].data.meas.get_counts())

In [ ]:
check("A3")

🤔 **For the discussion:** a real coin flipped twice is still random. Why does the quantum coin land on 0 **every** time? (Write out $H\cdot\tfrac{1}{\sqrt2}(\lvert0\rangle+\lvert1\rangle)$: which amplitudes add, and which cancel?)

### A4 · Explore: how many shots do you need? *(not scored)*

Run the cell. The error $\lvert f - 0.5\rvert$ in the fraction of 1s shrinks like $1/\sqrt{N}$: 100× more shots for 10× more precision.

In [ ]:
flips = coin.measure_all(inplace=False)
for shots in [10, 100, 1_000, 10_000, 100_000]:
    c = sampler.run([flips], shots=shots).result()[0].data.meas.get_counts()
    f = c.get("1", 0) / shots
    print(f"{shots:>7} shots: fraction of 1s = {f:.4f}   error = {abs(f - 0.5):.4f}   (1/sqrt(N) = {1 / sqrt(shots):.4f})")

---
## Part B · The biased coin

The rotation $R_y(\theta)$ turns $\lvert0\rangle$ part of the way towards $\lvert1\rangle$:

$$
R_y(\theta)\lvert0\rangle = \cos\tfrac{\theta}{2}\,\lvert0\rangle + \sin\tfrac{\theta}{2}\,\lvert1\rangle
\quad\Longrightarrow\quad P(1) = \sin^2\tfrac{\theta}{2}.
$$

### B1 · Find the angle for $P(1) = \tfrac14$

**Input:** a number `theta_B1` (radians; `pi` is available). **Output:** a coin that lands on 1 a quarter of the time.

Solve $\sin^2\frac{\theta}{2} = \frac14$ on paper first.

<details><summary><b>💡 Click for a hint</b></summary>

$\sin\frac{\theta}{2} = \frac12$, so $\frac{\theta}{2} = 30^\circ = \frac{\pi}{6}$.

Write the answer in radians, e.g. `pi / 3`.

</details>

In [ ]:
theta_B1 = None   # YOUR CODE HERE

if theta_B1 is not None:
    biased = QuantumCircuit(1)
    biased.ry(theta_B1, 0)
    print(Statevector(biased).probabilities_dict())
    display(plot_histogram(sampler.run([biased.measure_all(inplace=False)], shots=4000).result()[0].data.meas.get_counts()))

In [ ]:
check("B1")

### B2 · A coin with any bias

**Input:** a probability `p` between 0 and 1. **Output:** `make_coin(p)` returns a 1-qubit circuit with $P(1) = p$ exactly.

Turn the formula around: $\sin^2\frac\theta2 = p \;\Rightarrow\; \theta = 2\arcsin\sqrt{p}$.

<details><summary><b>💡 Click for a hint</b></summary>

`asin` and `sqrt` are already imported.

`qc.ry(2 * asin(sqrt(p)), 0)`

</details>

In [ ]:
def make_coin(p):
    qc = QuantumCircuit(1)
    # YOUR CODE HERE
    return qc

for p in [0.1, 0.5, 0.9]:
    print(p, Statevector(make_coin(p)).probabilities_dict())

In [ ]:
check("B2")

### B3 · Reveal the hidden phase

These two coins are **both** perfectly fair:

$$
\lvert+\rangle = \tfrac{1}{\sqrt2}(\lvert0\rangle + \lvert1\rangle), \qquad \lvert-\rangle = \tfrac{1}{\sqrt2}(\lvert0\rangle - \lvert1\rangle).
$$

Measured directly, you can't tell them apart (run the first cell). The minus sign is a **relative phase**: invisible to measurement… unless you add one gate first.

**Input:** `reveal(qc)` adds **one** gate to a copy of `qc`. **Output:** `reveal(plus_coin)` always gives `0`, and `reveal(minus_coin)` always gives `1`.

<details><summary><b>💡 Click for a hint</b></summary>

Which gate turns $\lvert+\rangle$ into $\lvert0\rangle$? You used it twice in A3.

$H\lvert-\rangle = \lvert1\rangle$.

</details>

In [ ]:
plus_coin = QuantumCircuit(1)
plus_coin.h(0)
minus_coin = QuantumCircuit(1)
minus_coin.x(0)
minus_coin.h(0)

both = [c.measure_all(inplace=False) for c in (plus_coin, minus_coin)]
res = sampler.run(both, shots=1000).result()
print("|+> measured:", res[0].data.meas.get_counts())
print("|-> measured:", res[1].data.meas.get_counts())

In [ ]:
def reveal(qc):
    qc = qc.copy()
    # YOUR CODE HERE: add ONE gate
    return qc

print(Statevector(reveal(plus_coin)).probabilities_dict(), Statevector(reveal(minus_coin)).probabilities_dict())

In [ ]:
check("B3")

🤔 **For the discussion:** if measuring can't see the phase, why does it matter? (Hint: in A3 and B3, the phase decided which amplitudes cancelled. Quantum algorithms on Day 2 are built on exactly this.)

---
## Part C · Entangled pairs

### C1 · Bit-order warm-up

Qiskit writes bitstrings with **qubit 0 on the right**: `'01'` means $q_1 = 0$, $q_0 = 1$.

**Input:** a 2-qubit circuit `always01`. **Output:** it must give the bitstring `'01'` every time.

<details><summary><b>💡 Click for a hint</b></summary>

Which qubit must be 1? The **rightmost** character is $q_0$.

`always01.x(0)`

</details>

In [ ]:
always01 = QuantumCircuit(2)
# YOUR CODE HERE

print(Statevector(always01).probabilities_dict())

In [ ]:
check("C1")

### C2 · Entangle two coins

Make a quantum coin on $q_0$, then use it to control a CNOT on $q_1$:

$$
\lvert00\rangle \xrightarrow{H \text{ on } q_0} \tfrac{1}{\sqrt2}(\lvert00\rangle + \lvert01\rangle) \xrightarrow{\mathrm{CX}_{0\to1}} \tfrac{1}{\sqrt2}(\lvert00\rangle + \lvert11\rangle) = \lvert\Phi^+\rangle .
$$

**Input:** a 2-qubit circuit `bell`. **Output:** the Bell state $\lvert\Phi^+\rangle$.

<details><summary><b>💡 Click for a hint</b></summary>

Two gates: `bell.h(0)` then `bell.cx(0, 1)` (control first, target second).

</details>

In [ ]:
bell = QuantumCircuit(2)
# YOUR CODE HERE

display(bell.draw("mpl"))
Statevector(bell).draw("latex")

In [ ]:
check("C2")

### C3 · Sample the pair

**Input:** 1,000 shots of `bell`. **Output:**
* `bell_counts`: the counts dict
* `p_q0`: the fraction of shots where $q_0$ (the **rightmost** bit) is `1`
* `p_same`: the fraction of shots where the two bits are **equal**

<details><summary><b>💡 Stuck? Click for a hint</b></summary>

`bell_counts = sampler.run([bell.measure_all(inplace=False)], shots=1000).result()[0].data.meas.get_counts()`

`p_q0 = sum(n for b, n in bell_counts.items() if b[-1] == '1') / 1000`

`p_same = sum(n for b, n in bell_counts.items() if b[0] == b[1]) / 1000`

</details>

In [ ]:
bell_counts = None
p_q0 = None
p_same = None
# YOUR CODE HERE

print(bell_counts, p_q0, p_same)
if bell_counts:
    display(plot_histogram(bell_counts))

In [ ]:
check("C3")

### C4 · Correlations with the Estimator

The Estimator returns expectation values directly. For a Pauli $Z$, $\langle Z\rangle = P(0) - P(1)$: it's $+1$ if the bit is always 0, $-1$ if always 1, and $0$ for a fair coin. For $ZZ$, it's $+1$ when the two bits always **agree**.

Pauli strings use Qiskit order too: `"ZI"` is $Z$ on $q_1$, `"IZ"` is $Z$ on $q_0$.

**Input:** the observables `"ZI"`, `"IZ"`, `"ZZ"` **in that order**, and the **unmeasured** `bell` circuit. **Output:** `bell_evs`, an array of three numbers.

<details><summary><b>💡 Click for a hint</b></summary>

`observables = [SparsePauliOp('ZI'), SparsePauliOp('IZ'), SparsePauliOp('ZZ')]`

`bell_evs = estimator.run([(bell, observables)]).result()[0].data.evs`

</details>

In [ ]:
bell_evs = None
# YOUR CODE HERE

print(bell_evs)

In [ ]:
check("C4")

Each coin **alone** is perfectly random ($\langle Z\rangle = 0$), yet the pair is perfectly correlated ($\langle ZZ\rangle = 1$).

### C5 · All four Bell states

There are four maximally entangled two-qubit states. Build the other three, each starting from $\lvert00\rangle$ (Qiskit order, $\lvert q_1 q_0\rangle$):

$$
\lvert\Phi^-\rangle = \tfrac{1}{\sqrt2}(\lvert00\rangle - \lvert11\rangle), \quad
\lvert\Psi^+\rangle = \tfrac{1}{\sqrt2}(\lvert01\rangle + \lvert10\rangle), \quad
\lvert\Psi^-\rangle = \tfrac{1}{\sqrt2}(\lvert01\rangle - \lvert10\rangle).
$$

<details><summary><b>💡 Click for a hint</b></summary>

Start from your Bell circuit. An $X$ before the CNOT changes **which** bits agree; a $Z$ changes the **sign**.

$\Phi^-$: `h(0)`, `z(0)`, `cx(0, 1)`. $\Psi^+$: `h(0)`, `x(1)`, `cx(0, 1)`.

$\Psi^-$: combine both tricks, then check with `Statevector(...).draw('latex')`.

</details>

In [ ]:
phi_minus = QuantumCircuit(2)
# YOUR CODE HERE

psi_plus = QuantumCircuit(2)
# YOUR CODE HERE

psi_minus = QuantumCircuit(2)
# YOUR CODE HERE

for name, qc in [("Φ-", phi_minus), ("Ψ+", psi_plus), ("Ψ-", psi_minus)]:
    print(name, Statevector(qc).draw("latex_source"))

In [ ]:
check("C5")

### C6 · Quantum vs classical correlation

Here's a **classical** imitation of the Bell pair: a machine that secretly flips one fair coin and gives you **two copies** of the result. You also always see `00` or `11`, 50/50. Is a Bell pair just that?

Test both in a **different basis**, with $X\otimes X$ instead of $Z\otimes Z$. The classical machine is a 50/50 mix of the two circuits `c00` and `c11` below, so its expectation value is the **average** of theirs.

**Input:** use the helper `expval(circuit, "XX")`. **Output:**
* `zz_classical`, `xx_classical`: averages over `c00` and `c11`
* `xx_bell`: for your `bell` circuit

<details><summary><b>💡 Click for a hint</b></summary>

`zz_classical = (expval(c00, 'ZZ') + expval(c11, 'ZZ')) / 2`, and the same with `'XX'`.

`xx_bell = expval(bell, 'XX')`

</details>

In [ ]:
def expval(qc, pauli):
    return float(estimator.run([(qc, SparsePauliOp(pauli))]).result()[0].data.evs)

c00 = QuantumCircuit(2)                      # the machine's "both heads" outcome
c11 = QuantumCircuit(2); c11.x([0, 1])       # the machine's "both tails" outcome

In [ ]:
zz_classical = None
xx_classical = None
xx_bell = None
# YOUR CODE HERE

print(f"classical: <ZZ> = {zz_classical}, <XX> = {xx_classical}")
print(f"Bell pair: <ZZ> = {expval(bell, 'ZZ') if bell.num_qubits == 2 else None}, <XX> = {xx_bell}")

In [ ]:
check("C6")

🤔 **For the discussion:** both pairs agree perfectly when measured with $Z$. Only the Bell pair **also** agrees when measured with $X$. What does that tell you about whether the Bell pair's outcomes were "decided in advance"? And can Alice use her half of a Bell pair to send Bob a message?

---
## Part D · Bonus

### D1 · GHZ of any size

**Input:** a number of qubits `n`. **Output:** `make_ghz(n)` returns the $n$-qubit state $\tfrac{1}{\sqrt2}(\lvert0\cdots0\rangle + \lvert1\cdots1\rangle)$.

<details><summary><b>💡 Click for a hint</b></summary>

Your Bell circuit is `make_ghz(2)`. Add a loop of CNOTs: `for i in range(n - 1): qc.cx(i, i + 1)`.

</details>

In [ ]:
def make_ghz(n):
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

print(Statevector(make_ghz(4)).probabilities_dict())

In [ ]:
check("D1")

### D2 · Your Bell pair on a noisy chip

`FakeTorino` copies the gates and error rates of a real 133-qubit IBM processor. Real chips only run a few native gates, so first **transpile** the circuit, then run it with the Runtime Sampler. This is exactly the code you'll use on real hardware on Day 2.

**Input:** your `bell` circuit. **Output:** `wrong_fraction`, the fraction of shots that are `'01'` or `'10'`: results an ideal Bell pair never gives.

<details><summary><b>💡 Click for a hint</b></summary>

The transpile and run lines are given; you only compute `wrong_fraction` from `noisy_counts`.

`wrong_fraction = (noisy_counts.get('01', 0) + noisy_counts.get('10', 0)) / 1000`

</details>

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeTorino()
isa = generate_preset_pass_manager(optimization_level=1, backend=backend).run(bell.measure_all(inplace=False))
noisy_counts = Sampler(mode=backend).run([isa], shots=1000).result()[0].data.meas.get_counts()
print(noisy_counts)
plot_histogram(noisy_counts)

In [ ]:
wrong_fraction = None
# YOUR CODE HERE

print(wrong_fraction)

In [ ]:
check("D2")

🤔 **For the discussion:** where do the "impossible" results come from? Look at `isa.draw("mpl", idle_wires=False)` to see what the chip actually ran.

---
## 🏁 Scoreboard

Run this cell, take a **screenshot**, and post it in the event chat **before the walkthrough**.

📮 **Submit your completed lab by Sunday, Nov 22, 11:59 AM (Central Time):** [forms.gle/9h5PA4iqQg1wfGAcA](https://forms.gle/9h5PA4iqQg1wfGAcA). You have until then to finish every part.

In [ ]:
scoreboard()

---
*Qiskit Fall Fest 2026 @ UT San Antonio · Lab 1 by Van Tien Nguyen (UT San Antonio). Tested with Qiskit 2.6.2, qiskit-ibm-runtime 0.52.1, qiskit-aer 0.18.2.*